# Ablation : Preprocessing Depth (Full Pipeline vs Partial)

**Research Question:** Does each step in the preprocessing pipeline earn its place?

## Three Conditions

| Condition | Bandpass + Notch | Artifact Rejection | Baseline Correction |
|-----------|:---:|:---:|:---:|
| **Full** | ✓ | ✓ | ✓ |
| **Filter Only** | ✓ | ✗ | ✗ |
| **Artifact + Baseline Only** | ✗ | ✓ | ✓ |

## Methods Under Test
- **MAML** (Full-MAML) — best meta-learner from main study
- **Supervised** — K-shot MLP baseline

## Protocol
- LOSO evaluation, 26 subjects, K ∈ {5, 10, 20}, N_SEEDS = 3
- Primary metric: balanced accuracy (class-imbalance immune)
- Secondary: AUC-ROC
- Epoch counts logged per condition (filtering changes survival rate)

**Key hypothesis:** Bandpass filtering is the critical step for EEG ERP detection,
because low-frequency drift and high-frequency noise severely distort the
Ne/ERN morphology (~250 ms) and Pe (~320 ms) that define ErrP.

## 1. Imports and Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import signal as sp_signal, stats
from scipy.stats import wilcoxon
from scipy.linalg import sqrtm as mat_sqrt
from copy import deepcopy
import glob, re, os, sys, json, pickle, time, warnings, logging, random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

import mne
from mne.io import RawArray

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

logging.getLogger('mne').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 2. Configuration — Single Source of Truth

**All** hyperparameters live in `Config`. No execution cell overrides any field.
`Config.SFREQ`, `Config.N_CHANNELS`, `Config.N_TIMES`, and `Config.KERNEL_LENGTH`
are set at runtime after data loading (their values depend on the actual dataset).

Key design decisions encoded here:
- `SUBJECT_EMBED_DIM = 32` is used in SubjectEncoder, FiLM, and every call site  
- `K_SHOTS = [5, 10, 20]` — K=50 removed (not a few-shot regime)  
- `N_SEEDS = 3` — minimum for publishable variance estimates


In [ ]:
def _resolve_dataset_root(default="/kaggle/input/inria-bci-challenge/inria-bci-challenge"):
    candidates = [default]
    cwd = os.getcwd()
    candidates += [
        os.path.join(cwd, "Code", "inria-bci-challenge"),
        os.path.join(cwd, "inria-bci-challenge"),
        os.path.join(cwd, "..", "Code", "inria-bci-challenge"),
    ]
    seen, uniq = set(), []
    for p in candidates:
        n = os.path.normpath(p)
        if n not in seen:
            seen.add(n); uniq.append(n)
    for root in uniq:
        if (os.path.isfile(os.path.join(root, "TrainLabels.csv"))
                and os.path.isdir(os.path.join(root, "train"))):
            return root
    return os.path.normpath(default)


def _resolve_output_root(default="/kaggle/working/results_v2"):
    return default if os.path.isdir("/kaggle/working") else os.path.normpath(
        os.path.join(os.getcwd(), "Results", "results_v2"))


class Config:
    # ── Paths ──────────────────────────────────────────────────────────
    DATASET_ROOT  = _resolve_dataset_root()
    TRAIN_DIR     = os.path.join(DATASET_ROOT, "train")
    LABELS_FILE   = os.path.join(DATASET_ROOT, "TrainLabels.csv")

    OUTPUT_ROOT    = _resolve_output_root()
    RESULTS_DIR    = OUTPUT_ROOT
    FIGURES_DIR    = os.path.join(OUTPUT_ROOT, "figures")
    METRICS_DIR    = os.path.join(OUTPUT_ROOT, "metrics")
    CSV_DIR        = os.path.join(OUTPUT_ROOT, "csv")
    CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "checkpoints")

    # ── Preprocessing ──────────────────────────────────────────────────
    TMIN             = -0.2      # epoch start relative to feedback onset (s)
    TMAX             =  0.6      # epoch end relative to feedback onset (s)
    BASELINE         = (-0.2, 0.0)
    LOWCUT           =  1.0      # bandpass lower cutoff (Hz)
    HIGHCUT          = 40.0      # bandpass upper cutoff (Hz)
    NOTCH_FREQ       = 50.0      # power-line notch (Hz)
    FILTER_ORDER     =  4        # Butterworth order
    ART_THRESHOLD_UV = 100.0     # artifact rejection threshold µV (peak-to-peak)

    # ── Feature extraction ─────────────────────────────────────────────
    PCA_COMPONENTS = 32

    # ── Experiment ─────────────────────────────────────────────────────
    K_SHOTS         = [5, 10, 20]   # K=50 removed (not few-shot)
    N_SEEDS         = 3
    RANDOM_SEEDS    = [42, 123, 456]
    N_EVAL_EPISODES = 10

    # ── Meta-learning ──────────────────────────────────────────────────
    N_META_ITERATIONS = 2000
    META_BATCH_SIZE   = 4
    N_SUPPORT         = 10
    N_QUERY           = 40
    INNER_LR          = 0.01
    OUTER_LR          = 5e-4
    INNER_STEPS       = 5

    # ── Encoder architecture ───────────────────────────────────────────
    ENCODER_HIDDEN  = 256   # 3-layer MLP
    ENCODER_HIDDEN2 = 128
    ENCODER_OUTPUT  = 64
    DROPOUT         = 0.3

    # ── Subject-conditioned meta-learner ───────────────────────────────
    SUBJECT_EMBED_DIM = 32   # consistent across SubjectEncoder and FiLM

    # ── EEGNet ─────────────────────────────────────────────────────────
    EEGNET_F1      = 8
    EEGNET_D       = 2
    EEGNET_F2      = 16
    EEGNET_DROPOUT = 0.25
    # KERNEL_LENGTH, SFREQ, N_CHANNELS, N_TIMES set at runtime after data load

    # ── Device ─────────────────────────────────────────────────────────
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # ── Ablation conditions (used as string keys) ──────────────────────
    PREPROCESSING_CONDITIONS = [
        "Full",               # bandpass + notch + artifact rejection + baseline
        "FilterOnly",         # bandpass + notch only
        "ArtifactBaselineOnly" # artifact rejection + baseline only (no filter)
    ]

for d in [Config.RESULTS_DIR, Config.FIGURES_DIR, Config.METRICS_DIR,
          Config.CSV_DIR, Config.CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Dataset root: {Config.DATASET_ROOT}")
print(f"Output root : {Config.OUTPUT_ROOT}")
print(f"Device      : {Config.DEVICE}")
print(f"K_SHOTS     : {Config.K_SHOTS}")
print(f"N_SEEDS     : {Config.N_SEEDS}")


## 3. Determinism and Reproducibility

`set_seed` enforces full determinism across Python random, NumPy, PyTorch CPU/GPU,
CUDA algorithm selection, and the CUBLAS workspace.

`freeze_batchnorm` prevents BatchNorm running statistics from updating during
inner-loop adaptation where batch sizes (K=5–20) are too small for reliable
batch statistics.


In [ ]:
def set_seed(seed: int = 42) -> None:
    """Enforce full determinism across all random sources."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


def freeze_batchnorm(model: nn.Module) -> None:
    """Put all BatchNorm layers in eval mode and disable running stat updates.

    Essential during inner-loop adaptation: batch sizes of K=5-20 produce
    unreliable batch statistics that corrupt running mean/variance.
    """
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.eval()
            m.track_running_stats = False


set_seed(Config.RANDOM_SEEDS[0])
print("Determinism enforced.")


## 4. Metrics — Primary: Balanced Accuracy and AUC-ROC

**Why NOT raw accuracy as primary metric:**

ErrP datasets have ~75% correct / ~25% error trials.
A classifier that always predicts "correct" achieves ~75% raw accuracy
while learning nothing at all.

Balanced accuracy (average recall per class) and AUC-ROC are immune to
class imbalance and are the standard in EEG BCI literature (Yasemin et al. 2023).

Raw accuracy is computed and reported as secondary reference only.


In [ ]:
def compute_comprehensive_metrics(
        predictions: np.ndarray,
        labels: np.ndarray,
        probabilities: Optional[np.ndarray] = None
) -> Dict[str, Any]:
    """Compute all metrics. Primary: balanced_accuracy, auroc. Secondary: accuracy.

    Args:
        predictions  : Predicted labels (0 or 1).
        labels       : Ground-truth labels (0 or 1).
        probabilities: Predicted probabilities shape (N,2) or (N,). Optional for AUC.

    Returns:
        Dict with balanced_accuracy, auroc, accuracy, f1_score, confusion matrix.
    """
    predictions = np.asarray(predictions).ravel()
    labels      = np.asarray(labels).ravel()

    metrics: Dict[str, Any] = {
        'accuracy'          : float(accuracy_score(labels, predictions)),
        'balanced_accuracy' : float(balanced_accuracy_score(labels, predictions)),
        'f1_score'          : float(f1_score(labels, predictions,
                                             average='binary', zero_division=0.0)),
    }

    cm = confusion_matrix(labels, predictions, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    metrics.update({'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
                    'n_samples': len(labels),
                    'n_class_0': int(np.sum(labels == 0)),
                    'n_class_1': int(np.sum(labels == 1))})

    if probabilities is not None:
        probs = np.asarray(probabilities)
        p1 = probs[:, 1] if probs.ndim == 2 and probs.shape[1] == 2 else probs.ravel()
        try:
            metrics['auroc'] = float(roc_auc_score(labels, p1))
        except ValueError:
            metrics['auroc'] = float('nan')
    else:
        metrics['auroc'] = float('nan')

    return metrics


def compute_chance_level(labels: np.ndarray) -> Dict[str, float]:
    """Chance level = always predict majority class. Balanced acc = 0.5 by definition."""
    majority = int(np.bincount(labels).argmax())
    preds = np.full_like(labels, majority)
    return compute_comprehensive_metrics(preds, labels)


## 5. Results I/O — Checkpointing and CSV Export

After each LOSO fold, the per-subject result is saved to disk as JSON.
If training crashes at fold 20/26, re-running picks up from fold 21.

`save_per_subject_metrics` and `save_aggregate_metrics` produce CSVs
compatible with downstream statistical analysis.


In [ ]:
def _ckpt_path(output_dir: str, method: str, seed: int, subject: str) -> Path:
    p = Path(output_dir) / f"seed_{seed}" / "fold_checkpoints"
    p.mkdir(parents=True, exist_ok=True)
    return p / f"{method}_{subject}.json"


def _json_safe(obj):
    if isinstance(obj, (np.floating, float)):
        v = float(obj)
        return None if (v != v) else v   # NaN → None for JSON
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_subject_checkpoint(output_dir: str, method: str, seed: int,
                             subject: str, result: Dict) -> None:
    """Save single-subject result immediately after computation (crash-safe)."""
    path = _ckpt_path(output_dir, method, seed, subject)
    with open(path, 'w') as f:
        json.dump(result, f, default=_json_safe)


def load_subject_checkpoint(output_dir: str, condition: str, method: str, seed: int,
                             subject: str) -> Optional[Dict]:
    """Load previously saved fold result, or None if not found."""
    path = _ckpt_path(output_dir, condition, method, seed, subject)
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None


def save_per_subject_metrics(results: Dict, method_name: str, seed: int,
                              output_dir: str = Config.RESULTS_DIR) -> None:
    """Write per-subject × per-K metrics to CSV."""
    seed_dir = Path(output_dir) / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for sid, sd in results.get('subjects', {}).items():
        for k, km in sd.get('k_shots', {}).items():
            rows.append({'method': method_name, 'seed': seed, 'subject_id': sid,
                         'k_shots': k,
                         **{m: km.get(m, float('nan'))
                            for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc',
                                      'n_samples', 'n_class_0', 'n_class_1',
                                      'tp', 'fp', 'tn', 'fn']}})
    pd.DataFrame(rows).to_csv(seed_dir / f"{method_name}_per_subject.csv", index=False)


def save_aggregate_metrics(results: Dict, method_name: str, seed: int,
                            k_shots: List[int],
                            output_dir: str = Config.RESULTS_DIR) -> None:
    """Write aggregate (mean ± std across subjects) metrics to CSV."""
    seed_dir = Path(output_dir) / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for k in k_shots:
        vals = defaultdict(list)
        for sd in results.get('subjects', {}).values():
            km = sd.get('k_shots', {}).get(k, {})
            for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc']:
                v = km.get(m, float('nan'))
                if not (isinstance(v, float) and v != v):  # skip NaN
                    vals[m].append(v)
        row = {'method': method_name, 'seed': seed, 'k_shots': k}
        for m, vs in vals.items():
            row[f'{m}_mean'] = float(np.mean(vs)) if vs else float('nan')
            row[f'{m}_std']  = float(np.std(vs))  if vs else float('nan')
        rows.append(row)
    pd.DataFrame(rows).to_csv(seed_dir / f"{method_name}_aggregate.csv", index=False)

## 7. Dataset Loading

The INRIA BCI Challenge dataset: CSV files named `Data_S{N}_Sess{M}.csv`
for 26 subjects × 5 sessions. Labels from `TrainLabels.csv`.

`parse_label_id` extracts subject/session/feedback index from entries like
`S02_Sess01_FB013`. The feedback index is used for **index-based** epoch–label
alignment (not positional truncation, which is incorrect when any epoch is
dropped due to boundary effects).


In [ ]:
def parse_filename(filename: str) -> Tuple[Optional[str], Optional[int]]:
    """Extract (subject_id, session_num) from 'Data_S02_Sess01.csv'."""
    m = re.match(r'Data_S(\d+)_Sess(\d+)\.csv', filename)
    return (f"S{m.group(1)}", int(m.group(2))) if m else (None, None)


def index_dataset(data_dir: str) -> Dict[str, List[Tuple[int, str]]]:
    """Return {subject_id: [(session_num, filepath), ...]} sorted by session."""
    idx: Dict = defaultdict(list)
    for fp in glob.glob(os.path.join(data_dir, "*.csv")):
        sid, sess = parse_filename(os.path.basename(fp))
        if sid:
            idx[sid].append((sess, fp))
    for k in idx:
        idx[k].sort()
    return dict(idx)


def load_labels(labels_file: str) -> Dict[str, Dict[int, List[Tuple[int, int]]]]:
    """Parse TrainLabels.csv → {subject → {session → [(fb_idx, label)]}}."""
    df = pd.read_csv(labels_file)
    ld: Dict = defaultdict(lambda: defaultdict(list))
    for _, row in df.iterrows():
        m = re.match(r'S(\d+)_Sess(\d+)_FB(\d+)', str(row['IdFeedBack']))
        if not m:
            continue
        sid  = f"S{m.group(1)}"
        sess = int(m.group(2))
        fbi  = int(m.group(3))
        ld[sid][sess].append((fbi, int(row['Prediction'])))
    for sid in ld:
        for sess in ld[sid]:
            ld[sid][sess].sort(key=lambda x: x[0])
    return {k: dict(v) for k, v in ld.items()}


def build_dataset_mapping(train_index: Dict, labels_dict: Dict) -> Dict[str, List[Dict]]:
    """Join file index with label lists into a single mapping."""
    mapping = {}
    for sid, sessions in train_index.items():
        if sid not in labels_dict:
            continue
        mapping[sid] = []
        for sess, fp in sessions:
            if sess not in labels_dict[sid]:
                continue
            mapping[sid].append({'session_num': sess, 'filepath': fp,
                                  'labels': labels_dict[sid][sess]})
    return mapping


# ── Execute ──────────────────────────────────────────────────────────────
train_index     = index_dataset(Config.TRAIN_DIR)
labels_dict     = load_labels(Config.LABELS_FILE)
dataset_mapping = build_dataset_mapping(train_index, labels_dict)

print(f"Subjects found : {len(dataset_mapping)}")
for sid in sorted(dataset_mapping)[:3]:
    total = sum(len(s['labels']) for s in dataset_mapping[sid])
    print(f"  {sid}: {len(dataset_mapping[sid])} sessions, {total} labelled trials")


## 8. Preprocessing Pipeline — Core Helpers

These low-level helpers are shared across all three conditions.
The three condition-specific functions in the next cell each call
(or deliberately skip) these helpers.

In [ ]:
def load_continuous_eeg(filepath: str) -> Tuple[pd.DataFrame, float]:
    """Load CSV EEG file. Infer sampling frequency from Time column."""
    df = pd.read_csv(filepath)
    dt = float(np.median(np.diff(df['Time'].values[:200])))
    return df, 1.0 / dt


def get_eeg_channels(df: pd.DataFrame) -> List[str]:
    """Return column names excluding Time and FeedBackEvent."""
    return [c for c in df.columns if c not in ('Time', 'FeedBackEvent')]


def apply_filters_continuous(eeg_data: np.ndarray, sfreq: float,
                              lowcut: float, highcut: float,
                              notch_freq: float, order: int = 4) -> np.ndarray:
    """Apply notch then bandpass to continuous EEG (n_channels × n_samples).

    Uses zero-phase sosfiltfilt to avoid phase distortion.
    Applied BEFORE epoching to prevent boundary edge artifacts.

    Args:
        eeg_data  : Shape (n_channels, n_samples), raw µV.
        sfreq     : Sampling frequency in Hz.
        lowcut    : Bandpass lower cutoff (Hz).
        highcut   : Bandpass upper cutoff (Hz).
        notch_freq: Power-line notch (Hz).
        order     : Butterworth filter order.
    """
    nyq = sfreq / 2.0
    out = eeg_data.copy()

    # 1. Notch filter
    nl, nh = (notch_freq - 1.0) / nyq, (notch_freq + 1.0) / nyq
    if 0 < nl < 1 and 0 < nh < 1:
        sos = sp_signal.butter(order, [nl, nh], btype='bandstop', output='sos')
        out = sp_signal.sosfiltfilt(sos, out, axis=1)

    # 2. Bandpass filter
    lo, hi = lowcut / nyq, min(highcut / nyq, 0.9999)
    if 0 < lo < hi < 1:
        sos = sp_signal.butter(order, [lo, hi], btype='band', output='sos')
        out = sp_signal.sosfiltfilt(sos, out, axis=1)

    return out


def detect_feedback_events(df: pd.DataFrame) -> np.ndarray:
    """Return sample indices of 0→1 transitions in FeedBackEvent column."""
    ev = df['FeedBackEvent'].values
    return np.where(np.diff(np.concatenate([[0], ev])) == 1)[0]


def create_epochs(eeg_data: np.ndarray, event_indices: np.ndarray,
                  sfreq: float, tmin: float, tmax: float
                  ) -> Tuple[np.ndarray, np.ndarray, List[int]]:
    """Slice epochs from filtered continuous EEG.

    Returns:
        epochs_data     : (n_valid, n_ch, n_times)
        times           : (n_times,) time axis in seconds
        valid_positions : list mapping epoch index → original event position
    """
    n_before = int(abs(tmin) * sfreq)
    n_after  = int(tmax * sfreq)
    n_total  = eeg_data.shape[1]
    epochs, valid_pos = [], []
    for pos, idx in enumerate(event_indices):
        s, e = idx - n_before, idx + n_after
        if s >= 0 and e <= n_total:
            epochs.append(eeg_data[:, s:e])
            valid_pos.append(pos)
    arr   = np.array(epochs) if epochs else np.empty((0, eeg_data.shape[0], n_before + n_after))
    times = np.linspace(tmin, tmax, n_before + n_after, endpoint=False)
    return arr, times, valid_pos


def align_epochs_with_labels_by_index(
        epochs_data: np.ndarray,
        valid_positions: List[int],
        session_labels: List[Tuple[int, int]]
) -> Tuple[np.ndarray, np.ndarray]:
    """Align epochs to labels using the feedback index — NOT positional truncation.

    Positional truncation (min(n_epochs, n_labels)) is WRONG:
    if any epoch is dropped at a boundary, all subsequent labels shift silently.

    This function matches epoch i to the label whose event position equals
    valid_positions[i], using the ordered feedback list as the position reference.
    """
    pos_to_label = {pos: lbl for pos, (_, lbl) in enumerate(session_labels)}
    aligned_e, aligned_l = [], []
    for ep_i, orig_pos in enumerate(valid_positions):
        if orig_pos in pos_to_label:
            aligned_e.append(epochs_data[ep_i])
            aligned_l.append(pos_to_label[orig_pos])
    if not aligned_e:
        return np.empty((0,) + epochs_data.shape[1:]), np.empty(0, dtype=int)
    return np.array(aligned_e), np.array(aligned_l, dtype=int)


def reject_artifacts(epochs_data: np.ndarray,
                     threshold_uv: float = 100.0) -> np.ndarray:
    """Return boolean mask (True = keep) for epochs below peak-to-peak threshold.

    Applied on RAW µV BEFORE baseline correction or z-score.
    The 100 µV threshold is standard in ERP literature (Luck 2014).
    """
    ptp = np.ptp(epochs_data, axis=2).max(axis=1)   # max ptp across channels
    return ptp <= threshold_uv


def baseline_correct(epochs_data: np.ndarray, times: np.ndarray,
                     baseline: Tuple[float, float] = (-0.2, 0.0)) -> np.ndarray:
    """Subtract per-epoch per-channel mean of the baseline window.

    Applied AFTER artifact rejection on CLEAN epochs.
    Computed independently for each epoch and each channel.
    """
    t0, t1 = baseline
    mask = (times >= t0) & (times <= t1)
    assert mask.sum() > 0, f"Baseline window {baseline} is empty (times range: {times[[0,-1]]})"
    bl_mean = epochs_data[:, :, mask].mean(axis=2, keepdims=True)  # (N, C, 1)
    return epochs_data - bl_mean

print("Core preprocessing helpers defined.")


## 9. Three Condition-Specific Preprocessing Functions

### Design
- `apply_filters_continuous` is **never modified** — we only skip *calling* it
- Conditions differ only in which pipeline steps are active
- Each function logs per-subject epoch counts and rejection stats

```
┌─────────────────────────────────────────────────────────┐
│ Step                     │ Full │ FilterOnly │ Art+BL   │
├─────────────────────────────────────────────────────────┤
│ Bandpass + Notch filter  │  ✓   │     ✓      │   ✗      │
│ Epoch extraction         │  ✓   │     ✓      │   ✓      │
│ Index-based label align  │  ✓   │     ✓      │   ✓      │
│ Artifact rejection 100µV │  ✓   │     ✗      │   ✓      │
│ Baseline correction      │  ✓   │     ✗      │   ✓      │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
# ── Condition 1: FULL pipeline (unchanged from main notebook) ──────────────

def process_subject_data_full(subject_id: str,
                               session_infos: List[Dict]) -> Optional[Dict]:
    """
    FULL pipeline:
      continuous filter (bandpass + notch)
      → epoch → index-align
      → artifact rejection (100 µV)
      → baseline correction
    """
    all_epochs, all_labels = [], []
    sfreqs: List[float] = []
    ch_names_ref = times_ref = None
    total_raw = total_rejected = 0

    for sess_info in session_infos:
        try:
            df, sfreq = load_continuous_eeg(sess_info['filepath'])
            ch_names  = get_eeg_channels(df)
            raw_data  = df[ch_names].values.T

            # ── STEP 1: filter continuous signal ──
            raw_filt = apply_filters_continuous(
                raw_data, sfreq,
                Config.LOWCUT, Config.HIGHCUT, Config.NOTCH_FREQ, Config.FILTER_ORDER)

            # ── STEP 2: epoch ──
            events = detect_feedback_events(df)
            epo, times, valid_pos = create_epochs(
                raw_filt, events, sfreq, Config.TMIN, Config.TMAX)
            if len(epo) == 0:
                continue

            # ── STEP 3: index-align ──
            epo_al, lbl_al = align_epochs_with_labels_by_index(
                epo, valid_pos, sess_info['labels'])
            if len(epo_al) == 0:
                continue

            # ── STEP 4: artifact rejection ──
            mask = reject_artifacts(epo_al, Config.ART_THRESHOLD_UV)
            total_raw      += len(epo_al)
            total_rejected += int((~mask).sum())
            epo_clean = epo_al[mask]
            lbl_clean = lbl_al[mask]
            if len(epo_clean) == 0:
                continue

            # ── STEP 5: baseline correction ──
            epo_bc = baseline_correct(epo_clean, times, Config.BASELINE)

            all_epochs.append(epo_bc)
            all_labels.append(lbl_clean)
            sfreqs.append(sfreq)
            ch_names_ref = ch_names
            times_ref    = times

        except Exception as exc:
            print(f"  [{subject_id}] session {sess_info.get('session_num','?')} failed: {exc}")

    if not all_epochs:
        return None

    epochs_cat = np.concatenate(all_epochs, axis=0).astype(np.float32)
    labels_cat = np.concatenate(all_labels, axis=0).astype(int)
    rej_rate   = 100.0 * total_rejected / max(total_raw, 1)
    cc = np.bincount(labels_cat, minlength=2)
    print(f"  {subject_id}: {len(epochs_cat)} epochs "
          f"(rej {total_rejected}/{total_raw}, {rej_rate:.1f}%) "
          f"| cls0={cc[0]} cls1={cc[1]}")

    return {'epochs': epochs_cat, 'labels': labels_cat,
            'ch_names': ch_names_ref, 'times': times_ref,
            'sfreq': float(np.mean(sfreqs)),
            'n_rejected': total_rejected, 'n_raw': total_raw,
            'condition': 'Full'}


# ── Condition 2: FILTER ONLY (bandpass + notch; skip artifact rejection and baseline) ─

def process_subject_data_filter_only(subject_id: str,
                                      session_infos: List[Dict]) -> Optional[Dict]:
    """
    FILTER ONLY pipeline:
      continuous filter (bandpass + notch)   ← active
      → epoch → index-align
      [artifact rejection]                   ← SKIPPED
      [baseline correction]                  ← SKIPPED

    Note: apply_filters_continuous is NOT modified; it is called normally.
    Artifact rejection and baseline correction are simply not called.
    """
    all_epochs, all_labels = [], []
    sfreqs: List[float] = []
    ch_names_ref = times_ref = None
    total_raw = 0  # no rejection in this condition

    for sess_info in session_infos:
        try:
            df, sfreq = load_continuous_eeg(sess_info['filepath'])
            ch_names  = get_eeg_channels(df)
            raw_data  = df[ch_names].values.T

            # ── STEP 1: filter continuous signal ── (ACTIVE)
            raw_filt = apply_filters_continuous(
                raw_data, sfreq,
                Config.LOWCUT, Config.HIGHCUT, Config.NOTCH_FREQ, Config.FILTER_ORDER)

            # ── STEP 2: epoch ──
            events = detect_feedback_events(df)
            epo, times, valid_pos = create_epochs(
                raw_filt, events, sfreq, Config.TMIN, Config.TMAX)
            if len(epo) == 0:
                continue

            # ── STEP 3: index-align ──
            epo_al, lbl_al = align_epochs_with_labels_by_index(
                epo, valid_pos, sess_info['labels'])
            if len(epo_al) == 0:
                continue

            # ── STEP 4: artifact rejection ── SKIPPED
            # reject_artifacts(...) not called — all epochs retained
            total_raw += len(epo_al)

            # ── STEP 5: baseline correction ── SKIPPED
            # baseline_correct(...) not called — raw filtered signal used as-is

            all_epochs.append(epo_al)    # unrejected, uncorrected
            all_labels.append(lbl_al)
            sfreqs.append(sfreq)
            ch_names_ref = ch_names
            times_ref    = times

        except Exception as exc:
            print(f"  [{subject_id}] session {sess_info.get('session_num','?')} failed: {exc}")

    if not all_epochs:
        return None

    epochs_cat = np.concatenate(all_epochs, axis=0).astype(np.float32)
    labels_cat = np.concatenate(all_labels, axis=0).astype(int)
    cc = np.bincount(labels_cat, minlength=2)
    print(f"  {subject_id}: {len(epochs_cat)} epochs "
          f"(no rejection) | cls0={cc[0]} cls1={cc[1]}")

    return {'epochs': epochs_cat, 'labels': labels_cat,
            'ch_names': ch_names_ref, 'times': times_ref,
            'sfreq': float(np.mean(sfreqs)),
            'n_rejected': 0, 'n_raw': total_raw,
            'condition': 'FilterOnly'}


# ── Condition 3: ARTIFACT + BASELINE ONLY (no filter) ─────────────────────

def process_subject_data_artifact_baseline_only(subject_id: str,
                                                 session_infos: List[Dict]) -> Optional[Dict]:
    """
    ARTIFACT + BASELINE ONLY pipeline:
      [continuous filter]                    ← SKIPPED (apply_filters_continuous not called)
      → epoch (from RAW unfiltered signal)
      → index-align
      artifact rejection (100 µV)            ← active
      baseline correction                    ← active

    Note: apply_filters_continuous is NOT modified — it just isn't called here.
    Epoching is performed directly on the raw µV signal.
    """
    all_epochs, all_labels = [], []
    sfreqs: List[float] = []
    ch_names_ref = times_ref = None
    total_raw = total_rejected = 0

    for sess_info in session_infos:
        try:
            df, sfreq = load_continuous_eeg(sess_info['filepath'])
            ch_names  = get_eeg_channels(df)
            raw_data  = df[ch_names].values.T

            # ── STEP 1: continuous filter ── SKIPPED
            # apply_filters_continuous(...) not called — epoch raw signal directly
            raw_for_epoching = raw_data   # unfiltered

            # ── STEP 2: epoch ──
            events = detect_feedback_events(df)
            epo, times, valid_pos = create_epochs(
                raw_for_epoching, events, sfreq, Config.TMIN, Config.TMAX)
            if len(epo) == 0:
                continue

            # ── STEP 3: index-align ──
            epo_al, lbl_al = align_epochs_with_labels_by_index(
                epo, valid_pos, sess_info['labels'])
            if len(epo_al) == 0:
                continue

            # ── STEP 4: artifact rejection ── (ACTIVE — on raw µV as required)
            mask = reject_artifacts(epo_al, Config.ART_THRESHOLD_UV)
            total_raw      += len(epo_al)
            total_rejected += int((~mask).sum())
            epo_clean = epo_al[mask]
            lbl_clean = lbl_al[mask]
            if len(epo_clean) == 0:
                continue

            # ── STEP 5: baseline correction ── (ACTIVE)
            epo_bc = baseline_correct(epo_clean, times, Config.BASELINE)

            all_epochs.append(epo_bc)
            all_labels.append(lbl_clean)
            sfreqs.append(sfreq)
            ch_names_ref = ch_names
            times_ref    = times

        except Exception as exc:
            print(f"  [{subject_id}] session {sess_info.get('session_num','?')} failed: {exc}")

    if not all_epochs:
        return None

    epochs_cat = np.concatenate(all_epochs, axis=0).astype(np.float32)
    labels_cat = np.concatenate(all_labels, axis=0).astype(int)
    rej_rate   = 100.0 * total_rejected / max(total_raw, 1)
    cc = np.bincount(labels_cat, minlength=2)
    print(f"  {subject_id}: {len(epochs_cat)} epochs "
          f"(rej {total_rejected}/{total_raw}, {rej_rate:.1f}%) "
          f"| cls0={cc[0]} cls1={cc[1]}")

    return {'epochs': epochs_cat, 'labels': labels_cat,
            'ch_names': ch_names_ref, 'times': times_ref,
            'sfreq': float(np.mean(sfreqs)),
            'n_rejected': total_rejected, 'n_raw': total_raw,
            'condition': 'ArtifactBaselineOnly'}


# ── Dispatch map ──────────────────────────────────────────────────────────
PREPROC_FN = {
    'Full'               : process_subject_data_full,
    'FilterOnly'         : process_subject_data_filter_only,
    'ArtifactBaselineOnly': process_subject_data_artifact_baseline_only,
}

print("Three preprocessing condition functions defined:")
for k, fn in PREPROC_FN.items():
    print(f"  {k:25s} → {fn.__name__}")

## 10. Run Preprocessing Under All Three Conditions

Each condition produces its own `preprocessed_subjects_data` dict.
Epoch counts differ because:
- **Full vs FilterOnly**: FilterOnly retains all epochs (no 100 µV gate) → more epochs
- **Full vs ArtifactBaselineOnly**: unfiltered signal has larger amplitude swings
  → artifact rejection rejects more epochs than in the filtered case

In [ ]:
# ── Run all three preprocessing conditions ────────────────────────────────

all_preprocessed: Dict[str, Dict[str, Dict]] = {}   # condition → {sid → data}
condition_stats: Dict[str, Dict] = {}               # condition → summary stats

for condition in Config.PREPROCESSING_CONDITIONS:
    print(f"\n{'='*60}")
    print(f"Condition: {condition}")
    print('='*60)

    fn = PREPROC_FN[condition]
    ppd: Dict[str, Dict] = {}
    failed: List[str] = []

    for sid in tqdm(sorted(dataset_mapping.keys()), desc=condition):
        result = fn(sid, dataset_mapping[sid])
        if result is not None:
            ppd[sid] = result
        else:
            failed.append(sid)
            print(f"  FAILED: {sid}")

    # ── Condition summary stats ──
    n_subjects = len(ppd)
    all_epoch_counts  = [len(ppd[s]['epochs']) for s in ppd]
    all_raw_counts    = [ppd[s]['n_raw'] for s in ppd]
    all_reject_counts = [ppd[s]['n_rejected'] for s in ppd]

    total_epochs    = sum(all_epoch_counts)
    total_raw       = sum(all_raw_counts)
    total_rejected  = sum(all_reject_counts)
    rej_rate_pct    = 100.0 * total_rejected / max(total_raw, 1)

    condition_stats[condition] = {
        'n_subjects'    : n_subjects,
        'total_epochs'  : total_epochs,
        'total_raw'     : total_raw,
        'total_rejected': total_rejected,
        'rejection_rate': rej_rate_pct,
        'mean_epochs_per_subject': float(np.mean(all_epoch_counts)) if all_epoch_counts else 0,
        'std_epochs_per_subject' : float(np.std(all_epoch_counts))  if all_epoch_counts else 0,
        'failed_subjects': failed,
    }

    print(f"\n  Subjects    : {n_subjects}")
    print(f"  Total epochs: {total_epochs} "
          f"(raw={total_raw}, rejected={total_rejected}, rate={rej_rate_pct:.1f}%)")
    print(f"  Per subject : {np.mean(all_epoch_counts):.1f} ± {np.std(all_epoch_counts):.1f}")
    if failed:
        print(f"  FAILED      : {failed}")

    all_preprocessed[condition] = ppd

# ── Print comparison table ──────────────────────────────────────────────
print("\n" + "="*65)
print(f"{'Condition':<25} {'Subjects':>8} {'Epochs':>8} {'Rejected%':>10} {'Mean±Std':>16}")
print("-"*65)
for cond, st in condition_stats.items():
    print(f"{cond:<25} {st['n_subjects']:>8} {st['total_epochs']:>8} "
          f"{st['rejection_rate']:>9.1f}% "
          f"  {st['mean_epochs_per_subject']:>5.0f}±{st['std_epochs_per_subject']:.0f}")
print("="*65)

# ── Set runtime Config fields from the Full condition ────────────────────
# Use Full as the reference — all conditions share the same recording hardware
# (same sfreq, same channels, same epoch length). Only epoch *count* differs.
first_subj = list(all_preprocessed['Full'].values())[0]
Config.SFREQ      = first_subj['sfreq']
Config.N_CHANNELS = len(first_subj['ch_names'])
Config.N_TIMES    = first_subj['epochs'].shape[2]

print(f"\nRuntime Config fields set from 'Full' condition:")
print(f"  sfreq      = {Config.SFREQ:.1f} Hz")
print(f"  n_channels = {Config.N_CHANNELS}")
print(f"  n_times    = {Config.N_TIMES}")

## 11. Feature Extraction and LOSO-Isolated PCA

**Two processing branches:**

**Branch A — Classical ML** (MLP encoders, MAML, ProtoNets, Matching, Reptile)  
Features: flattened temporal waveform (n_channels × n_times) → PCA(32).  
Temporal features outperform bandpower for ERP detection because discriminative
information is carried by waveform morphology (Ne/ERN at ~250 ms, Pe at ~320 ms),
not spectral power (Yasemin et al. 2023).

**Branch B — Deep / Riemannian** (EEGNet, Riemannian baselines)  
Raw filtered waveforms (n_channels × n_times). EEGNet learns temporal and
spatial filters end-to-end. Riemannian methods use covariance matrices.

### LOSO PCA isolation
PCA is fitted on the N-1 training subjects' features only.
Test subject features are transformed using the training PCA.
No test subject statistics contaminate the PCA transformation.


In [ ]:
def extract_temporal_features(epochs: np.ndarray) -> np.ndarray:
    """Flatten (N, C, T) → (N, C*T). Temporal waveform features for meta-learning."""
    n, c, t = epochs.shape
    return epochs.reshape(n, c * t)


def extract_features_all_subjects(ppd: Dict[str, Dict]) -> Dict[str, Dict]:
    """Extract and return temporal features for all subjects."""
    out = {}
    for sid, data in tqdm(sorted(ppd.items()), desc="Feature extraction"):
        out[sid] = {'subject_id': sid,
                    'features': extract_temporal_features(data['epochs']).astype(np.float32),
                    'labels':   data['labels']}
    return out


class PCAReducer:
    """Fit StandardScaler + whitened PCA on training data; transform any data."""
    def __init__(self, n_components: int = 32, seed: int = 42):
        self.scaler    = StandardScaler()
        self.pca       = PCA(n_components=n_components, whiten=True, random_state=seed)
        self.is_fitted = False

    def fit(self, X: np.ndarray) -> 'PCAReducer':
        self.pca.fit(self.scaler.fit_transform(X))
        self.is_fitted = True
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return self.pca.transform(self.scaler.transform(X)).astype(np.float32)


def apply_pca_loso(subjects_features: Dict[str, Dict],
                   n_components: int = Config.PCA_COMPONENTS) -> Dict[str, Dict]:
    """Build LOSO PCA splits. Each fold's PCA fitted on N-1 training subjects.

    Each fold contains:
      test            : {features (PCA-reduced), labels}
      pca             : fitted PCAReducer for this fold
      train_subjects  : list of training subject IDs
      subjects_features: reference to raw feature dict (for training transforms)
    """
    sids = sorted(subjects_features.keys())
    splits = {}
    for test_sid in tqdm(sids, desc="LOSO PCA"):
        train_sids = [s for s in sids if s != test_sid]
        train_X    = np.concatenate([subjects_features[s]['features'] for s in train_sids])
        reducer    = PCAReducer(n_components=n_components).fit(train_X)
        splits[test_sid] = {
            'test'             : {'features': reducer.transform(subjects_features[test_sid]['features']),
                                  'labels'  : subjects_features[test_sid]['labels']},
            'pca'              : reducer,
            'train_subjects'   : train_sids,
            'subjects_features': subjects_features,
        }
    return splits


all_subjects_features: Dict[str, Dict] = {}
all_loso_splits: Dict[str, Dict]       = {}
INPUT_DIM_PER_CONDITION: Dict[str, int] = {}

for condition in Config.PREPROCESSING_CONDITIONS:
    print(f"\nBuilding features for: {condition}")
    ppd  = all_preprocessed[condition]
    sf   = extract_features_all_subjects(ppd)
    ls   = apply_pca_loso(sf, Config.PCA_COMPONENTS)

    all_subjects_features[condition] = sf
    all_loso_splits[condition]        = ls

    first_fold = ls[sorted(ls)[0]]
    input_dim  = first_fold['test']['features'].shape[1]
    INPUT_DIM_PER_CONDITION[condition] = input_dim
    print(f"  PCA dim={input_dim}, folds={len(ls)}, "
          f"train subjects per fold={len(first_fold['train_subjects'])}")


## 12. LOSO Isolation Verification

Run before any training to confirm no data leakage.

In [ ]:
def balanced_episode(features: np.ndarray, labels: np.ndarray,
                      n_support: int, n_query: int
                      ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Shared balanced episode sampler used by MAML and verification."""
    unique = np.unique(labels)
    sp_per = max(1, n_support // len(unique))
    qp_per = max(1, n_query   // len(unique)) if n_query > 0 else 0
    sxl, syl, qxl, qyl = [], [], [], []
    for cls in unique:
        idx = np.where(labels == cls)[0]
        n   = sp_per + qp_per
        if len(idx) < n:
            sp = max(1, len(idx) // 2); qp = len(idx) - sp
        else:
            sp, qp = sp_per, qp_per
        perm = np.random.permutation(idx)
        sxl.append(features[perm[:sp]]);       syl.append(labels[perm[:sp]])
        qxl.append(features[perm[sp:sp+qp]]); qyl.append(labels[perm[sp:sp+qp]])
    return (np.concatenate(sxl), np.concatenate(syl),
            np.concatenate(qxl), np.concatenate(qyl))


def verify_loso_isolation(loso_splits: Dict, condition: str,
                           verbose: bool = True) -> bool:
    all_ok = True
    for test_sid, fold in loso_splits.items():
        if test_sid in fold['train_subjects']:
            print(f"FAIL [{condition}]: {test_sid} in own training set!")
            all_ok = False
        tdim = fold['test']['features'].shape[1]
        pdim = fold['pca'].pca.n_components_
        if tdim != pdim:
            print(f"FAIL [{condition}]: {test_sid} dim mismatch {tdim} != {pdim}")
            all_ok = False
        if np.any(~np.isfinite(fold['test']['features'])):
            print(f"FAIL [{condition}]: {test_sid} contains NaN/Inf!")
            all_ok = False

    # Support/query disjointness spot-check
    first_sid = sorted(loso_splits.keys())[0]
    test_X = loso_splits[first_sid]['test']['features']
    test_y = loso_splits[first_sid]['test']['labels']
    for k in Config.K_SHOTS:
        sx, sy, qx, qy = balanced_episode(test_X, test_y, k, 40)
        sx_set = {tuple(x.round(6)) for x in sx}
        qx_set = {tuple(x.round(6)) for x in qx}
        if sx_set & qx_set:
            print(f"FAIL [{condition}]: K={k} support/query overlap!")
            all_ok = False

    if all_ok and verbose:
        print(f"  OK  [{condition}] LOSO isolation verified: {len(loso_splits)} folds.")
    return all_ok


print("Verifying LOSO isolation for all conditions...")
for condition in Config.PREPROCESSING_CONDITIONS:
    verify_loso_isolation(all_loso_splits[condition], condition)

## 13. Baselines

### Supervised Baseline (K-shot MLP)
Train a fresh MLP from scratch on exactly K balanced examples per test subject.  
Uses early stopping (patience=20) and class-weighted loss.

This is the standard non-meta comparison baseline.
Max 200 epochs (the original used 2000, leading to severe overfitting on K=5–20 examples).

### Zero-Shot Baseline
Computed inside each meta-learning training loop: apply the meta-initialized
model to each test subject WITHOUT any K-shot adaptation.
Quantifies how much performance comes from the meta-initialization alone.

### Chance Level
Always predict the majority class (~75% correct trials).
Balanced accuracy = 0.5 exactly (by definition of balanced accuracy).


In [ ]:
def run_supervised_baseline_loso(
        loso_splits: Dict, k_shots: List[int],
        condition: str,
        hidden_dim: int = 128, lr: float = 1e-3, n_epochs: int = 200,
        device: str = str(Config.DEVICE), seed: int = 42,
        output_dir: str = Config.RESULTS_DIR) -> Dict:
    """
    Supervised few-shot baseline with LOSO evaluation.

    For each subject and each K:
      1. Sample K/n_classes examples per class (balanced support set).
      2. Train 3-layer MLP with LayerNorm + GELU for up to n_epochs.
      3. Early stopping (patience=20) on support set loss.
      4. Evaluate on remaining test subject examples.
    Condition label is threaded through for checkpointing isolation.
    """
    set_seed(seed)
    all_results: Dict = {}

    for test_sid, fold in tqdm(loso_splits.items(), desc=f"Supervised [{condition}]"):
        ckpt = load_subject_checkpoint(output_dir, condition, 'Supervised', seed, test_sid)
        if ckpt is not None:
            all_results[test_sid] = ckpt
            continue
        
        test_X    = fold['test']['features']
        test_y    = fold['test']['labels']
        input_dim = test_X.shape[1]
        subject_results = {'subject_id': test_sid, 'k_shots': {}}

        for k in k_shots:
            fold_seed = seed + hash(test_sid) % 10000 + k
            set_seed(fold_seed)

            # ── Balanced K-shot support set ─────────────────────────────
            unique_cls = np.unique(test_y)
            k_per_cls  = max(1, k // len(unique_cls))
            supp_idx = []
            for cls in unique_cls:
                cls_idx = np.where(test_y == cls)[0]
                n = min(k_per_cls, len(cls_idx))
                supp_idx.extend(np.random.choice(cls_idx, size=n, replace=False))
            supp_idx  = np.array(supp_idx)
            query_idx = np.array([i for i in range(len(test_X)) if i not in supp_idx])
            assert len(set(supp_idx) & set(query_idx)) == 0

            train_X = test_X[supp_idx]; train_y = test_y[supp_idx]
            eval_X  = test_X[query_idx]; eval_y  = test_y[query_idx]
            if len(eval_y) == 0:
                continue

            # ── Class weights ─────────────────────────────────────────
            counts = np.bincount(train_y, minlength=2).astype(float)
            counts = np.where(counts == 0, 1.0, counts)
            w = torch.FloatTensor(2.0 / (counts * counts.sum())).to(device)

            # ── Model: 3-layer MLP with LayerNorm + GELU ──────────────
            model = nn.Sequential(
                nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(0.3),
                nn.Linear(hidden_dim, 64),        nn.LayerNorm(64),         nn.GELU(), nn.Dropout(0.3),
                nn.Linear(64, 2)
            ).to(device)

            opt  = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
            crit = nn.CrossEntropyLoss(weight=w)
            tx   = torch.FloatTensor(train_X).to(device)
            ty   = torch.LongTensor(train_y).to(device)

            # ── Train with early stopping ──────────────────────────────
            best_loss, patience_ctr, best_state = float('inf'), 0, None
            model.train()
            for _ in range(n_epochs):
                opt.zero_grad()
                loss = crit(model(tx), ty)
                loss.backward(); opt.step()
                if loss.item() < best_loss - 1e-5:
                    best_loss   = loss.item()
                    best_state  = deepcopy(model.state_dict())
                    patience_ctr = 0
                else:
                    patience_ctr += 1
                if patience_ctr >= 20:
                    break
            if best_state is not None:
                model.load_state_dict(best_state)

            # ── Evaluate ──────────────────────────────────────────────
            model.eval()
            with torch.no_grad():
                lgts  = model(torch.FloatTensor(eval_X).to(device))
                probs = F.softmax(lgts, dim=1).cpu().numpy()
                preds = lgts.argmax(dim=1).cpu().numpy()
            subject_results['k_shots'][k] = compute_comprehensive_metrics(preds, eval_y, probs)

            del model
            if 'cuda' in str(device):
                torch.cuda.empty_cache()

        all_results[test_sid] = subject_results
        save_subject_checkpoint(
            output_dir, condition, 'Supervised', seed, test_sid, subject_results)

    return {'method': 'Supervised', 'condition': condition,
            'subjects': all_results, 'k_shots': k_shots, 'seed': seed}


## 13. Models — EEGEncoder and MetaEEGClassifier

In [ ]:
class EEGEncoder(nn.Module):
    """
    Meta-learned EEG feature encoder.

    Architecture: input → 256 → 128 → 64 (representation)
    Each hidden layer: Linear → LayerNorm → GELU → Dropout(0.3)
    Final layer:       Linear  [NO activation, NO dropout — unconstrained embedding]

    Rationale:
    - LayerNorm: makes activations amplitude-invariant across subjects
    - GELU: better gradient flow than ReLU for meta-learning outer loops
    - No final activation: unconstrained embedding (ReLU would discard ~50% of space)
    - Size 256→128→64: capacity without over-parameterization at 32-dim PCA input
    """
    def __init__(self, input_dim: int,
                 h1: int = 256, h2: int = 128, out_dim: int = 64,
                 dropout: float = 0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, h1)
        self.ln1 = nn.LayerNorm(h1)
        self.dp1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(h1, h2)
        self.ln2 = nn.LayerNorm(h2)
        self.dp2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(h2, out_dim)    # representation layer

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dp1(F.gelu(self.ln1(self.fc1(x))))
        x = self.dp2(F.gelu(self.ln2(self.fc2(x))))
        return self.fc3(x)   # no activation


class SimpleTaskHead(nn.Module):
    """Linear classifier on encoder representation. Adapted per-subject in inner loop."""
    def __init__(self, input_dim: int = 64, num_classes: int = 2):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


class MetaEEGClassifier(nn.Module):
    """Full model: EEGEncoder + SimpleTaskHead. Used by MAML, ANIL, Reptile."""
    def __init__(self, input_dim: int,
                 h1: int = 256, h2: int = 128, enc_out: int = 64,
                 dropout: float = 0.3, num_classes: int = 2):
        super().__init__()
        self.encoder   = EEGEncoder(input_dim, h1, h2, enc_out, dropout)
        self.task_head = SimpleTaskHead(enc_out, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.task_head(self.encoder(x))

    def get_repr(self, x: torch.Tensor) -> torch.Tensor:
        """Encoder representation without classification."""
        return self.encoder(x)


# ── Sanity check ─────────────────────────────────────────────────────────
_m = MetaEEGClassifier(32)
_param_keys = [n for n, _ in _m.named_parameters()]
print("MetaEEGClassifier parameter keys:")
for k in _param_keys:
    print(f"  {k}")
assert 'task_head.fc.weight' in _param_keys, "task_head.fc.weight missing!"
assert 'task_head.fc.bias'   in _param_keys, "task_head.fc.bias missing!"
del _m
print("\nEEGEncoder, SimpleTaskHead, MetaEEGClassifier defined and verified.")

## 14. MAML (Full-MAML)

Full-MAML: both encoder and task head adapt in the inner loop.
CosineAnnealingLR + gradient clipping + class-weighted CE + balanced episodes.

In [ ]:
class MAML_Encoder:
    """
    MAML / ANIL for EEG-based BCI personalization.

    Outer: Adam + CosineAnnealingLR + gradient clipping (max_norm=1.0)
    Inner: functional gradient updates (correct gradient flow)
    Loss : class-weighted cross-entropy (handles 3:1 ErrP imbalance)
    Episodes: balanced (K/2 per class support and query)
    """
    def __init__(self, input_dim: int,
                 h1: int = Config.ENCODER_HIDDEN,
                 h2: int = Config.ENCODER_HIDDEN2,
                 enc_out: int = Config.ENCODER_OUTPUT,
                 num_classes: int = 2,
                 inner_lr: float = Config.INNER_LR,
                 outer_lr: float = Config.OUTER_LR,
                 inner_steps: int = Config.INNER_STEPS,
                 n_meta_iterations: int = Config.N_META_ITERATIONS,
                 freeze_encoder_inner: bool = False,
                 first_order: bool = True,
                 device: str = str(Config.DEVICE)):
        self.inner_lr   = inner_lr
        self.inner_steps = inner_steps
        self.freeze_enc  = freeze_encoder_inner
        self.first_order = first_order
        self.device      = torch.device(device)

        self.meta_model = MetaEEGClassifier(
            input_dim, h1, h2, enc_out, Config.DROPOUT, num_classes).to(self.device)
        self.meta_opt   = optim.Adam(self.meta_model.parameters(), lr=outer_lr)
        self.scheduler  = optim.lr_scheduler.CosineAnnealingLR(
            self.meta_opt, T_max=n_meta_iterations, eta_min=outer_lr * 0.01)

    # ── Utility: class weights ────────────────────────────────────────────
    @staticmethod
    def _class_weights(labels: np.ndarray, device: torch.device,
                       n_classes: int = 2) -> torch.Tensor:
        counts = np.bincount(labels, minlength=n_classes).astype(float)
        counts = np.where(counts == 0, 1.0, counts)
        w = 1.0 / counts; w = w / w.sum() * n_classes
        return torch.FloatTensor(w).to(device)

    # ── Functional forward pass ──────────────────────────────────────────
    def _functional_forward(self, x: torch.Tensor, params: dict) -> torch.Tensor:
        """Forward pass using parameter dict (enables functional gradient flow)."""
        h = F.dropout(F.gelu(F.layer_norm(
                F.linear(x, params['encoder.fc1.weight'], params['encoder.fc1.bias']),
                [params['encoder.fc1.weight'].shape[0]])),
                p=Config.DROPOUT, training=self.meta_model.training)
        h = F.dropout(F.gelu(F.layer_norm(
                F.linear(h, params['encoder.fc2.weight'], params['encoder.fc2.bias']),
                [params['encoder.fc2.weight'].shape[0]])),
                p=Config.DROPOUT, training=self.meta_model.training)
        h = F.linear(h, params['encoder.fc3.weight'], params['encoder.fc3.bias'])
        return F.linear(h, params['task_head.fc.weight'], params['task_head.fc.bias'])

    # ── Functional inner loop ─────────────────────────────────────────────
    def _inner_loop(self, sx: torch.Tensor, sy: torch.Tensor,
                    params: dict, cw: torch.Tensor) -> Tuple[dict, float]:
        """Inner-loop gradient adaptation using functional gradients.

        Uses SHALLOW dict copy — tensors are shared, not cloned.
        This preserves the computational graph from query loss back to meta-params.
        """
        adapted   = dict(params)   # shallow copy — DO NOT use deepcopy here
        to_update = ({k: v for k, v in adapted.items() if 'task_head' in k}
                     if self.freeze_enc else adapted)
        total_loss = 0.0
        for _ in range(self.inner_steps):
            logits = self._functional_forward(sx, adapted)
            loss   = F.cross_entropy(logits, sy, weight=cw)
            grads  = torch.autograd.grad(
                loss, to_update.values(),
                create_graph=not self.first_order, allow_unused=True)
            for (name, _), g in zip(list(to_update.items()), grads):
                if g is not None:
                    g_ = g.detach() if self.first_order else g
                    adapted[name] = adapted[name] - self.inner_lr * g_
            to_update = {k: adapted[k] for k in to_update}
            total_loss += loss.item()
        return adapted, total_loss / self.inner_steps

    def meta_update(self, tasks: list) -> dict:
        """Outer-loop meta-update across a batch of tasks."""
        self.meta_opt.zero_grad()
        params = {n: p for n, p in self.meta_model.named_parameters()}
        meta_losses, q_accs = [], []

        for sx, sy, qx, qy in tasks:
            sx = torch.FloatTensor(sx).to(self.device)
            sy = torch.LongTensor(sy).to(self.device)
            qx = torch.FloatTensor(qx).to(self.device)
            qy = torch.LongTensor(qy).to(self.device)
            cw = self._class_weights(sy.cpu().numpy(), self.device)

            adapted, _ = self._inner_loop(sx, sy, params, cw)
            q_logits   = self._functional_forward(qx, adapted)
            q_cw       = self._class_weights(qy.cpu().numpy(), self.device)
            q_loss     = F.cross_entropy(q_logits, qy, weight=q_cw)
            meta_losses.append(q_loss)
            with torch.no_grad():
                q_accs.append((q_logits.argmax(1) == qy).float().mean().item())

        meta_loss = torch.stack(meta_losses).mean()
        meta_loss.backward()
        nn.utils.clip_grad_norm_(self.meta_model.parameters(), max_norm=1.0)
        self.meta_opt.step()
        self.scheduler.step()
        return {'meta_loss': meta_loss.item(), 'query_acc': float(np.mean(q_accs))}

    def adapt_and_evaluate(self, support_X: np.ndarray, support_y: np.ndarray,
                            query_X: np.ndarray, query_y: np.ndarray) -> dict:
        """Test-time adaptation and evaluation."""
        adapted_model = deepcopy(self.meta_model)
        freeze_batchnorm(adapted_model)

        if self.freeze_enc:
            adapt_params = list(adapted_model.task_head.parameters())
            for p in adapted_model.encoder.parameters():
                p.requires_grad_(False)
        else:
            adapt_params = list(adapted_model.parameters())

        sx  = torch.FloatTensor(support_X).to(self.device)
        sy  = torch.LongTensor(support_y).to(self.device)
        cw  = self._class_weights(support_y, self.device)
        opt = optim.SGD(adapt_params, lr=self.inner_lr)

        adapted_model.train()
        for _ in range(self.inner_steps):
            opt.zero_grad()
            F.cross_entropy(adapted_model(sx), sy, weight=cw).backward()
            opt.step()

        adapted_model.eval()
        with torch.no_grad():
            lgts  = adapted_model(torch.FloatTensor(query_X).to(self.device))
            probs = F.softmax(lgts, 1).cpu().numpy()
            preds = lgts.argmax(1).cpu().numpy()
        del adapted_model
        return compute_comprehensive_metrics(preds, query_y, probs)


def train_maml_loso(
        loso_splits: dict, k_shots: list,
        condition: str,
        freeze_encoder_inner: bool = False,
        n_meta_iterations: int = Config.N_META_ITERATIONS,
        meta_batch_size: int = Config.META_BATCH_SIZE,
        n_support: int = Config.N_SUPPORT, n_query: int = Config.N_QUERY,
        inner_lr: float = Config.INNER_LR, outer_lr: float = Config.OUTER_LR,
        inner_steps: int = Config.INNER_STEPS,
        device: str = str(Config.DEVICE), seed: int = 42,
        output_dir: str = Config.RESULTS_DIR,
        method_name: str = 'MAML') -> dict:
    """LOSO MAML training. Condition label scopes checkpoints."""
    set_seed(seed)
    all_results: dict = {}

    for fold_idx, (test_sid, fold) in enumerate(
            tqdm(loso_splits.items(), desc=f"{method_name} LOSO")):
        fold_seed = seed + fold_idx * 13
        set_seed(fold_seed)

        ckpt = load_subject_checkpoint(output_dir,condition, method_name, seed, test_sid)
        if ckpt is not None:
            all_results[test_sid] = ckpt
            continue

        train_sids = fold['train_subjects']
        train_dict = {s: {'features': fold['pca'].transform(
                              fold['subjects_features'][s]['features']),
                          'labels'  : fold['subjects_features'][s]['labels']}
                      for s in train_sids}
        input_dim = fold['test']['features'].shape[1]

        agent = MAML_Encoder(input_dim=input_dim, freeze_encoder_inner=freeze_encoder_inner,
                             inner_lr=inner_lr, outer_lr=outer_lr, inner_steps=inner_steps,
                             n_meta_iterations=n_meta_iterations,
                             first_order=freeze_encoder_inner, device=device)

        # ── Meta-training ─────────────────────────────────────────────────
        for it in range(n_meta_iterations):
            batch = np.random.choice(train_sids,
                                     size=min(meta_batch_size, len(train_sids)),
                                     replace=False)
            tasks = []
            for s in batch:
                f, l = train_dict[s]['features'], train_dict[s]['labels']
                if len(f) < n_support + n_query:
                    continue
                sx, sy, qx, qy = balanced_episode(f, l, n_support, n_query)
                tasks.append((sx, sy, qx, qy))
            if tasks:
                agent.meta_update(tasks)

        # ── Zero-shot baseline ─────────────────────────────────────────────
        test_X = fold['test']['features']
        test_y = fold['test']['labels']
        agent.meta_model.eval()
        with torch.no_grad():
            lgts = agent.meta_model(torch.FloatTensor(test_X).to(agent.device))
            zs_probs = F.softmax(lgts, 1).cpu().numpy()
            zs_preds = lgts.argmax(1).cpu().numpy()
        zero_shot = compute_comprehensive_metrics(zs_preds, test_y, zs_probs)

        # ── K-shot evaluation ──────────────────────────────────────────────
        subject_results = {'subject_id': test_sid,
                           'zero_shot': zero_shot, 'k_shots': {}}
        for k in k_shots:
            eps = []
            for ep in range(Config.N_EVAL_EPISODES):
                set_seed(fold_seed + k * 100 + ep)
                sx, sy, qx, qy = MAML_Encoder._balanced_episode(
                    test_X, test_y, k, min(len(test_y) - k, 200))
                if len(qy) == 0:
                    continue
                eps.append(agent.adapt_and_evaluate(sx, sy, qx, qy))
            agg = {}
            for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc']:
                vals = [e[m] for e in eps if not (e[m] != e[m])]
                agg[m] = float(np.mean(vals)) if vals else float('nan')
            subject_results['k_shots'][k] = agg

        all_results[test_sid] = subject_results
        save_subject_checkpoint(output_dir, condition, method_name, seed, test_sid, subject_results)
        del agent
        if 'cuda' in str(device):
            torch.cuda.empty_cache()

    return {'method': method_name, 'condition': condition, 
            'subjects': all_results, 'k_shots': k_shots, 'seed': seed}

print("MAML and Supervised functions defined.")

## 15. Main Execution — All Conditions × 2 Methods × 3 Seeds

For each of the 3 preprocessing conditions:
1. Run Supervised baseline (K-shot MLP)
2. Run MAML (Full-MAML, best meta-learner)

Results are checkpointed per-condition so partial runs can resume.

In [ ]:

DEVICE     = str(Config.DEVICE)
K_SHOTS    = Config.K_SHOTS
SEEDS      = Config.RANDOM_SEEDS
OUTPUT_DIR = Config.RESULTS_DIR

# condition → seed → method → results
all_results_by_condition: Dict[str, Dict[int, Dict[str, dict]]] = {
    c: {} for c in Config.PREPROCESSING_CONDITIONS
}

for condition in Config.PREPROCESSING_CONDITIONS:
    loso_splits = all_loso_splits[condition]
    
    for current_seed in SEEDS:
        print(f"\n{'='*60}")
        print(f"Condition: {condition}  |  Seed: {current_seed}")
        print('='*65)
        set_seed(current_seed)
        seed_results: Dict[str, dict] = {}

        # ── Supervised baseline ──────────────────────────────────────────────
        seed_results['Supervised'] = run_supervised_baseline_loso(
            loso_splits, K_SHOTS, condition=condition, seed=current_seed, output_dir=OUTPUT_DIR)
        save_per_subject_metrics(seed_results['Supervised'], 'Supervised', current_seed, OUTPUT_DIR)
        save_aggregate_metrics(seed_results['Supervised'], 'Supervised', current_seed, K_SHOTS, OUTPUT_DIR)
        print(f"  Supervised done")

    # ── Full MAML ────────────────────────────────────────────────────────
        seed_results['Full-MAML'] = train_maml_loso(
            loso_splits, K_SHOTS, freeze_encoder_inner=False,
            device=DEVICE, seed=current_seed, output_dir=OUTPUT_DIR, method_name='Full-MAML')
        save_per_subject_metrics(seed_results['Full-MAML'], 'Full-MAML', current_seed, OUTPUT_DIR)
        save_aggregate_metrics(seed_results['Full-MAML'], 'Full-MAML', current_seed, K_SHOTS, OUTPUT_DIR)
        print(f"  Full-MAML done")


print("\n✓ All conditions × seeds complete.")

## 16. Results — Performance Tables

### Primary metric: Balanced Accuracy
### Secondary: AUC-ROC

In [ ]:
def extract_metric_values(results_by_seed: Dict[int, Dict[str, dict]],
                           method: str, k: int,
                           metric: str = 'balanced_accuracy') -> List[float]:
    """Collect all per-subject metric values across seeds for a given method and K."""
    vals = []
    for sr in results_by_seed.values():
        if method not in sr:
            continue
        for sd in sr[method].get('subjects', {}).values():
            v = sd.get('k_shots', {}).get(k, {}).get(metric)
            if v is not None and not (isinstance(v, float) and v != v):
                vals.append(float(v))
    return vals


def print_ablation_table(all_results_by_condition: Dict,
                          k_shots: List[int],
                          metric: str = 'balanced_accuracy') -> pd.DataFrame:
    """Print and return the 3-condition × 2-method × 3-K results table."""
    methods    = ['Supervised', 'MAML']
    conditions = Config.PREPROCESSING_CONDITIONS
    rows = []

    for condition in conditions:
        for method in methods:
            row = {'Condition': condition, 'Method': method}
            for k in k_shots:
                vals = extract_metric_values(
                    all_results_by_condition[condition], method, k, metric)
                if vals:
                    row[f'K={k} mean'] = round(float(np.mean(vals)), 4)
                    row[f'K={k} std']  = round(float(np.std(vals)),  4)
                else:
                    row[f'K={k} mean'] = float('nan')
                    row[f'K={k} std']  = float('nan')
            rows.append(row)

    df = pd.DataFrame(rows)

    metric_label = metric.replace('_', ' ').title()
    print(f"\n{'='*75}")
    print(f"Ablation 3 — Preprocessing Depth | Metric: {metric_label}")
    print(f"Chance level = 0.500  |  Seeds={len(list(all_results_by_condition.values())[0])}")
    print('='*75)
    print(df.to_string(index=False))
    print('='*75)

    # Save
    out_path = Path(Config.CSV_DIR) / f"ablation3_{metric}.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return df


df_bacc = print_ablation_table(all_results_by_condition, K_SHOTS, 'balanced_accuracy')
df_auc  = print_ablation_table(all_results_by_condition, K_SHOTS, 'auroc')

## 17. Epoch Count Summary Table

Documents how each condition affects the data available for training.

In [ ]:
print("\n" + "="*75)
print("Ablation 3 — Epoch Counts by Preprocessing Condition")
print("="*75)
print(f"{'Condition':<25} {'Subj':>5} {'Total Epochs':>13} {'Rej Rate':>10} "
      f"{'Mean±Std / Subject':>20}")
print("-"*75)
for cond in Config.PREPROCESSING_CONDITIONS:
    st = condition_stats[cond]
    print(f"{cond:<25} {st['n_subjects']:>5} {st['total_epochs']:>13,} "
          f"  {st['rejection_rate']:>8.1f}% "
          f"  {st['mean_epochs_per_subject']:>6.0f} ± {st['std_epochs_per_subject']:.0f}")
print("="*75)

print("\nInterpretation:")
print("  FilterOnly > Full (epochs): no 100µV gate → more epochs survive")
print("  ArtifactBaselineOnly: unfiltered signal has larger amplitude swings")
print("    → artifact rejection removes more epochs than in the filtered case")

# Save epoch stats
epoch_rows = []
for cond, st in condition_stats.items():
    epoch_rows.append({
        'condition'            : cond,
        'n_subjects'           : st['n_subjects'],
        'total_epochs'         : st['total_epochs'],
        'total_raw'            : st['total_raw'],
        'total_rejected'       : st['total_rejected'],
        'rejection_rate_pct'   : round(st['rejection_rate'], 2),
        'mean_epochs_per_subj' : round(st['mean_epochs_per_subject'], 1),
        'std_epochs_per_subj'  : round(st['std_epochs_per_subject'], 1),
    })
epoch_df = pd.DataFrame(epoch_rows)
epoch_df.to_csv(Path(Config.CSV_DIR) / "ablation3_epoch_counts.csv", index=False)
print(f"\nEpoch stats saved: {Config.CSV_DIR}/ablation3_epoch_counts.csv")

## 18. Statistical Tests — Full vs Each Partial Condition

Paired Wilcoxon signed-rank test on per-subject balanced accuracies.
Compares Full pipeline against FilterOnly and ArtifactBaselineOnly.

In [ ]:
from scipy.stats import norm as _norm


def paired_wilcoxon_conditions(
        results_condA: Dict[int, Dict[str, dict]],
        results_condB: Dict[int, Dict[str, dict]],
        method: str, k: int,
        metric: str = 'balanced_accuracy') -> Dict:
    """
    Paired Wilcoxon between two conditions for a given method and K.
    Pairs are (same subject, same seed) across conditions.
    """
    pairsA, pairsB = [], []
    for seed in sorted(results_condA.keys() & results_condB.keys()):
        srA = results_condA[seed].get(method, {}).get('subjects', {})
        srB = results_condB[seed].get(method, {}).get('subjects', {})
        for sid in sorted(set(srA) & set(srB)):
            vA = srA[sid].get('k_shots', {}).get(k, {}).get(metric)
            vB = srB[sid].get('k_shots', {}).get(k, {}).get(metric)
            if (vA is not None and vB is not None
                    and not (vA != vA) and not (vB != vB)):
                pairsA.append(float(vA))
                pairsB.append(float(vB))

    if len(pairsA) < 2:
        return {'error': f'Only {len(pairsA)} pairs', 'n': len(pairsA)}

    a1, a2 = np.array(pairsA), np.array(pairsB)
    try:
        stat, p = wilcoxon(a1, a2, alternative='two-sided')
    except ValueError as e:
        return {'error': str(e), 'n': len(pairsA)}

    n    = len(pairsA)
    diff = a1 - a2
    z    = _norm.ppf(1 - p / 2) * np.sign(np.mean(diff))
    r    = z / np.sqrt(n)
    interp = ('negligible' if abs(r) < 0.1 else 'small' if abs(r) < 0.3
              else 'medium' if abs(r) < 0.5 else 'large')
    return {'n': n, 'mean_A': float(np.mean(a1)), 'mean_B': float(np.mean(a2)),
            'mean_diff': float(np.mean(diff)),
            'statistic': float(stat), 'p_value': float(p),
            'significant': bool(p < 0.05),
            'effect_r': float(r), 'effect_interp': interp}


print("\n" + "="*75)
print("Statistical Tests: Full vs Partial Conditions (Wilcoxon signed-rank)")
print(f"Metric: Balanced Accuracy  |  α=0.05")
print("="*75)

stat_rows = []
partial_conditions = ['FilterOnly', 'ArtifactBaselineOnly']

for method in ['Supervised', 'MAML']:
    for cond_B in partial_conditions:
        for k in K_SHOTS:
            res = paired_wilcoxon_conditions(
                all_results_by_condition['Full'],
                all_results_by_condition[cond_B],
                method, k)
            row = {'method': method, 'Full_vs': cond_B, 'K': k, **res}
            stat_rows.append(row)
            if 'error' not in res:
                sig = "*" if res['significant'] else " "
                print(f"  {method:12s} Full vs {cond_B:22s} K={k:2d}: "
                      f"Full={res['mean_A']:.4f} vs Partial={res['mean_B']:.4f} "
                      f"Δ={res['mean_diff']:+.4f}  p={res['p_value']:.4f}{sig} "
                      f"r={res['effect_r']:+.3f} ({res['effect_interp']})")

print("="*75)

stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(Path(Config.CSV_DIR) / "ablation3_statistical_tests.csv", index=False)
print(f"\nStatistical tests saved: {Config.CSV_DIR}/ablation3_statistical_tests.csv")

## 19. Visualization — Preprocessing Depth vs Performance

In [ ]:
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300,
                     'font.family': 'sans-serif', 'axes.linewidth': 1.2})

CONDITION_COLORS = {
    'Full'               : '#2ECC71',
    'FilterOnly'         : '#3498DB',
    'ArtifactBaselineOnly': '#E74C3C',
}
CONDITION_LABELS = {
    'Full'               : 'Full Pipeline',
    'FilterOnly'         : 'Filter Only',
    'ArtifactBaselineOnly': 'Artifact + Baseline Only',
}
METHOD_LINESTYLES = {'Supervised': '--', 'MAML': '-'}
METHOD_MARKERS    = {'Supervised': 'D', 'MAML': 'o'}


def plot_preprocessing_ablation(all_results_by_condition: Dict,
                                 metric: str = 'balanced_accuracy',
                                 save_path: Optional[str] = None) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
    methods   = ['Supervised', 'MAML']
    ks        = sorted(K_SHOTS)

    for ax, method in zip(axes, methods):
        for condition in Config.PREPROCESSING_CONDITIONS:
            means, stds = [], []
            for k in ks:
                vals = extract_metric_values(
                    all_results_by_condition[condition], method, k, metric)
                means.append(float(np.mean(vals)) if vals else float('nan'))
                stds.append(float(np.std(vals))  if vals else 0.0)

            color  = CONDITION_COLORS[condition]
            label  = CONDITION_LABELS[condition]
            ls     = METHOD_LINESTYLES[method]
            marker = METHOD_MARKERS[method]

            ax.plot(ks, means, color=color, linestyle=ls, marker=marker,
                    linewidth=2.2, markersize=9, label=label, zorder=3)
            ax.fill_between(ks,
                            [m - s for m, s in zip(means, stds)],
                            [m + s for m, s in zip(means, stds)],
                            alpha=0.15, color=color, zorder=2)

        ax.axhline(0.5, color='grey', linestyle=':', linewidth=1.5,
                   alpha=0.6, label='Chance (0.50)', zorder=1)
        ax.set_xlabel('K-shots', fontsize=13, fontweight='bold')
        ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=13, fontweight='bold')
        ax.set_title(f'{method}', fontsize=14, fontweight='bold')
        ax.legend(fontsize=9, framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(ks)

    fig.suptitle(
        f'Ablation 3: Preprocessing Depth  |  Metric: {metric.replace("_"," ").title()}\n'
        f'(mean ± std over 26 subjects × {len(SEEDS)} seeds)',
        fontsize=13, fontweight='bold')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show(); plt.close(fig)


def plot_condition_comparison_bar(all_results_by_condition: Dict,
                                   k: int = 10,
                                   metric: str = 'balanced_accuracy',
                                   save_path: Optional[str] = None) -> None:
    """Bar chart comparing all 3 conditions × 2 methods at a fixed K."""
    methods    = ['Supervised', 'MAML']
    conditions = Config.PREPROCESSING_CONDITIONS
    x          = np.arange(len(conditions))
    width      = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))

    for i, method in enumerate(methods):
        means, errors = [], []
        for condition in conditions:
            vals = extract_metric_values(
                all_results_by_condition[condition], method, k, metric)
            means.append(float(np.mean(vals)) if vals else float('nan'))
            errors.append(float(np.std(vals)) if vals else 0.0)

        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, means, width,
                      yerr=errors, capsize=5,
                      color=[CONDITION_COLORS[c] for c in conditions],
                      alpha=0.85 if method == 'MAML' else 0.55,
                      label=method,
                      edgecolor='black', linewidth=0.8)

        for bar, m in zip(bars, means):
            if not np.isnan(m):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                        f'{m:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.axhline(0.5, color='red', linestyle='--', linewidth=1.5,
               alpha=0.5, label='Chance (0.50)')
    ax.set_xticks(x)
    ax.set_xticklabels([CONDITION_LABELS[c] for c in conditions], fontsize=11)
    ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    ax.set_title(f'Ablation 3 — K={k}  |  {metric.replace("_"," ").title()}\n'
                 f'(mean ± std, 26 subjects × {len(SEEDS)} seeds)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show(); plt.close(fig)


# Generate plots
for metric in ['balanced_accuracy', 'auroc']:
    plot_preprocessing_ablation(
        all_results_by_condition, metric=metric,
        save_path=os.path.join(Config.FIGURES_DIR,
                               f'ablation3_{metric}_vs_k.png'))

for k_val in K_SHOTS:
    plot_condition_comparison_bar(
        all_results_by_condition, k=k_val, metric='balanced_accuracy',
        save_path=os.path.join(Config.FIGURES_DIR,
                               f'ablation3_bar_k{k_val}.png'))

print(f"\nAll figures saved to: {Config.FIGURES_DIR}")

In [ ]:
def print_key_findings(all_results_by_condition: Dict,
                        k_shots: List[int],
                        metric: str = 'balanced_accuracy') -> None:
    """
    Summarise the key findings of the ablation:
    - How much does filtering contribute vs artifact rejection?
    - Which step is critical for each method?
    """
    print("\n" + "="*75)
    print("ABLATION 3 KEY FINDINGS — Preprocessing Depth")
    print("="*75)

    for method in ['Supervised', 'MAML']:
        print(f"\n▶ {method}")
        print(f"  {'K':>4}  {'Full':>8}  {'FiltOnly':>8}  {'Art+BL':>8}  "
              f"{'Δ(Full-FO)':>11}  {'Δ(Full-AB)':>11}")
        print("  " + "-"*65)

        for k in k_shots:
            v_full = extract_metric_values(
                all_results_by_condition['Full'], method, k, metric)
            v_fo   = extract_metric_values(
                all_results_by_condition['FilterOnly'], method, k, metric)
            v_ab   = extract_metric_values(
                all_results_by_condition['ArtifactBaselineOnly'], method, k, metric)

            m_full = float(np.mean(v_full)) if v_full else float('nan')
            m_fo   = float(np.mean(v_fo))   if v_fo   else float('nan')
            m_ab   = float(np.mean(v_ab))   if v_ab   else float('nan')

            delta_fo = m_full - m_fo   # positive = Full better than FilterOnly
            delta_ab = m_full - m_ab   # positive = Full better than Art+BL

            print(f"  K={k:>2}  {m_full:>8.4f}  {m_fo:>8.4f}  {m_ab:>8.4f}  "
                  f"  {delta_fo:>+9.4f}  {delta_ab:>+9.4f}")

    print("\n  Δ(Full - FilterOnly)   > 0 → baseline correction + artifact rejection add value")
    print("  Δ(Full - Art+BL Only)  > 0 → bandpass filtering adds value")
    print("  If Δ(Full - Art+BL) >> Δ(Full - FilterOnly): filtering is the critical step")
    print("  If similar: both contribute about equally")
    print("="*75)


print_key_findings(all_results_by_condition, K_SHOTS, 'balanced_accuracy')
print_key_findings(all_results_by_condition, K_SHOTS, 'auroc')

# Final combined CSV
print("\nBuilding final combined results CSV...")
final_rows = []
for condition in Config.PREPROCESSING_CONDITIONS:
    for method in ['Supervised', 'MAML']:
        for k in K_SHOTS:
            for metric in ['balanced_accuracy', 'auroc', 'f1_score']:
                vals = extract_metric_values(
                    all_results_by_condition[condition], method, k, metric)
                final_rows.append({
                    'condition': condition, 'method': method, 'k_shots': k,
                    'metric'   : metric,
                    'mean'     : round(float(np.mean(vals)), 6) if vals else float('nan'),
                    'std'      : round(float(np.std(vals)),  6) if vals else float('nan'),
                    'n'        : len(vals),
                })

final_df = pd.DataFrame(final_rows)
final_path = Path(Config.CSV_DIR) / "ablation3_final_summary.csv"
final_df.to_csv(final_path, index=False)
print(f"Saved: {final_path}")